# 01. Xe II: вероятности переходов (A) и ветвления для всех линий 6p → 6s, 5d

Без A нельзя получить сечение линии: σ_линии = σ(возбуждения верхнего уровня) · A_ul/ΣA.
В NIST у большинства 6p-уровней известны 0–4 ветви из 7–21. Здесь A считаются в DBSR
(`dbsr_mult3` + `dbsr_dmat3`) на той же мишени, что у Ванга, с номерами уровней Ванга.

Расчёт только структурный (без рассеяния): мишень (dbsr_hf + mchf) и E1/M1/E2 для всех
пар состояний. Идёт в фоне, результат — `transitions_E1.csv` и т. д.

Проверки: (1) энергии против NIST; (2) A против NIST (T8043 и др.); (3) отношение
скорость/длина (калибровки) — близко к 1 для надёжных линий; (4) времена жизни 6p
против измеренных (впишите свои значения из литературы).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
job = nb.Job(RUNS / "xe2_wang", "xe2_wang", levels=XLSX, stage="transitions", cores=CORES,
             multipoles="E1,E2")
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

## Энергии мишени против NIST

In [ ]:
W = RUNS / "xe2_wang"
print((W / "target_table.txt").read_text())

## A: DBSR против NIST

In [ ]:
ref = reference.read_xlsx(XLSX)
elv = [lv.energy_cm for lv in ref.levels]
tr = nb.read_csv(W / "transitions_E1.csv")
calc = {(int(r["upper_no"]), int(r["lower_no"])): r for r in tr if r["upper_no"] and r["lower_no"]}
pairs = []
for x in nb.nist_lines(NIST / "nist_XeII.csv"):
    u, l = nb.level_number(elv, x["Ek"]), nb.level_number(elv, x["Ei"])
    if x["A"] and (u, l) in calc:
        r = calc[(u, l)]
        pairs.append((x["wl"], x["A"], nb.fnum(r["A_exp"]), nb.fnum(r["gauge_ratio"]) or np.nan, u, l))
pairs = np.array(pairs)
print(f"{len(pairs)} линий с A в NIST")
for wl, an, ac, gr, u, l in pairs[np.argsort(pairs[:, 0])]:
    print(f"{wl:9.3f}  {int(u):3d}->{int(l):3d}  NIST {an:9.2e}  DBSR {ac:9.2e}  DBSR/NIST {ac/an:5.2f}  vel/len {gr:5.2f}")
plt.loglog(pairs[:, 1], pairs[:, 2], "o"); x = np.array([1e5, 3e8]); plt.plot(x, x, "k-", x, 2 * x, "k:", x, x / 2, "k:")
plt.xlabel("A NIST, с⁻¹"); plt.ylabel("A DBSR, с⁻¹"); plt.show()

## Ветвления и времена жизни 6p

`MEASURED_TAU_NS` — впишите измеренные времена жизни 6p-уровней Xe II из литературы
(номер уровня Ванга → нс), они попадут в сравнение.

In [ ]:
MEASURED_TAU_NS = {}           # например {26: 7.5, ...}: № уровня Ванга -> τ (нс)
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
for u in sorted(j for j in tot if 25 <= j <= 56 or j in (101, 105)):
    tau = 1e9 / tot[u]
    top = ", ".join(f"{l}:{b:.2f}" for l, b in sorted(br[u].items(), key=lambda kv: -kv[1])[:5])
    meas = f"  изм. {MEASURED_TAU_NS[u]} нс" if u in MEASURED_TAU_NS else ""
    print(f"{u:3d} {ref.label(u):28s} τ = {tau:6.2f} нс{meas}   ветвления: {top}")

## Ветвления для линий из списка (lines.csv) → `xe2_branching.csv`

In [ ]:
import csv
lines = [r for r in nb.read_csv(NIST / "lines.csv") if r["ion"] == "Xe II" and r["class"] in ("A", "B")]
out = []
for r in lines:
    u, l = int(r["upper_no"]), (int(r["lower_no"]) if r["lower_no"] else None)
    b = br.get(u, {}).get(l) if l else None
    out.append(dict(wl_nm=r["wl_nm"], upper_no=u, lower_no=l, upper=r["upper"], lower=r["lower"],
                    A_dbsr=A.get(u, {}).get(l), A_nist=r["A_nist"], branching=b, tau_ns=1e9 / tot[u] if u in tot else None,
                    observed=r.get("observed", ""), role=r.get("role", "")))
with open(W / "xe2_branching.csv", "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=list(out[0])); wr.writeheader(); wr.writerows(out)
for o in out:
    if o["role"]:
        print(f"{o['wl_nm']:>8}  {o['upper_no']:3d}->{o['lower_no']}  BR = {o['branching'] or float('nan'):.3f}  "
              f"A = {o['A_dbsr'] or float('nan'):.2e} (NIST {o['A_nist'] or '—'})  {o['role']}")

## E2: времена жизни метастабилей

5d-уровни с J ≥ 7/2 не распадаются E1 в 5p⁵; их сток — E2 в другие чётные уровни (и M1,
который здесь не считается: `dbsr_dmat3` падает на M1 для c-файлов; для CRM это
несущественно — такие распады на порядки медленнее пролёта).

In [ ]:
for kind in ("E2",):
    f = W / f"transitions_{kind}.csv"
    if f.exists():
        _, t, _ = nb.branching_from_csv(f)
        for u in sorted(t)[:12]:
            print(kind, u, ref.label(u), f"A_sum = {t[u]:.3e} с⁻¹  (τ = {1 / t[u]:.3e} с)")